# 🤖 Minion AI — Training Notebook

Modernized GPT training stack.  Train on Colab → run locally on RTX 3050.

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4 or better)
2. Run cells top to bottom
3. Checkpoints save to Google Drive automatically
4. If session disconnects, **re-run all cells** — training resumes from the last checkpoint

---
**Mode A** (pretrain): Train Minion from scratch (~125M params, ~3-5 sessions)

**Mode B** (qlora_finetune): Fine-tune GPT-2 XL with LoRA adapters (~1 session)

## Cell 1 — Mount Google Drive
Checkpoints will be saved here and survive session disconnects.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_CKPT_DIR = '/content/drive/MyDrive/minion_ckpts'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f'Checkpoint directory: {DRIVE_CKPT_DIR}')
print('Existing checkpoints:', os.listdir(DRIVE_CKPT_DIR) or 'none yet')

Mounted at /content/drive
Checkpoint directory: /content/drive/MyDrive/minion_ckpts
Existing checkpoints: none yet


## Cell 2 — Install dependencies

In [2]:
# Install pinned requirements (train side)
# bitsandbytes: CUDA 12.x wheel is pre-installed on Colab T4
%pip install -q \
    torch>=2.2.0 \
    bitsandbytes>=0.43.0 \
    peft>=0.10.0 \
    transformers>=4.40.0 \
    accelerate>=0.28.0 \
    tokenizers>=0.19.0 \
    datasets>=2.18.0 \
    PyYAML>=6.0 \
    wandb>=0.16.0 \
    tqdm>=4.66.0 \
    sentencepiece>=0.2.0

print('Dependencies installed ✓')

Dependencies installed ✓


## Cell 3 — Clone / pull Minion AI repo

In [3]:
import os

REPO_URL  = 'https://github.com/mlwithharsh/Minion.git'  # ← update this
REPO_DIR  = '/content/minion'

if os.path.exists(REPO_DIR):
    print('Repo exists — pulling latest changes...')
    !git -C {REPO_DIR} pull
else:
    print('Cloning repo...')
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
print('Files:', os.listdir('.'))

Cloning repo...
Cloning into '/content/minion'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 39 (delta 7), reused 33 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 671.48 KiB | 15.26 MiB/s, done.
Resolving deltas: 100% (7/7), done.
Working directory: /content/minion
Files: ['requirements-infer.txt', 'model.py', 'train.py', 'data.py', 'README.md', 'tokenizer', 'inference', '.git', 'tokenizer_train.py', 'requirements-train.txt', 'config', 'notebooks']


## Cell 4 — Detect GPU
Minion AI auto-detects GPU properties and adjusts precision accordingly.
T4 → fp16 + GradScaler | A100 → bfloat16

In [4]:
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1024**3
    print(f'GPU:        {props.name}')
    print(f'VRAM:       {vram_gb:.1f} GB')
    print(f'Compute:    {props.major}.{props.minor}')
    print(f'bf16:       {torch.cuda.is_bf16_supported()}')
    print()
    if vram_gb < 12:
        print('⚠️  Less than 12 GB VRAM detected.')
        print('   Reduce batch_size or block_size if you get OOM.')
    elif vram_gb >= 40:
        print('🚀 A100/H100 detected — you can scale up to 350M or 1B+ params!')
        print('   Edit config: n_layer=24, n_head=16, n_embd=1024')
else:
    print('❌ No GPU detected — change runtime to GPU!')

GPU:        Tesla T4
VRAM:       14.6 GB
Compute:    7.5
bf16:       True



## Cell 5 — Train a custom tokenizer (Mode A only)
Skip this cell for Mode B (the base model's tokenizer is used automatically).

This streams FineWeb-Edu and trains a 32k BPE tokenizer.
Takes ~5–15 minutes depending on `max_docs`.

In [5]:
# ---- Configuration ----
TRAINING_MODE = 'pretrain'    # 'pretrain' or 'qlora_finetune'
TOKENIZER_DIR = 'tokenizer'
VOCAB_SIZE    = 32768

if TRAINING_MODE == 'pretrain':
    # Check if tokenizer already exists (from a previous session)
    import os
    if os.path.exists(f'{TOKENIZER_DIR}/tokenizer.json'):
        print(f'Tokenizer already exists at {TOKENIZER_DIR}/tokenizer.json — skipping training')
    else:
        print(f'Training {VOCAB_SIZE}-token BPE tokenizer from FineWeb-Edu...')
        !python tokenizer_train.py \
            --mode train \
            --hf_dataset HuggingFaceFW/fineweb-edu \
            --hf_dataset_config sample-10BT \
            --vocab_size {VOCAB_SIZE} \
            --algorithm bpe \
            --tokenizer_dir {TOKENIZER_DIR} \
            --max_docs 50000
        print('Tokenizer training complete ✓')
else:
    print('Mode B selected — using base model tokenizer (no training needed)')

Tokenizer already exists at tokenizer/tokenizer.json — skipping training


## Cell 6 — Configure training
Edit the config path and any overrides here.

In [6]:
# Choose your config:
#   Mode A pretrain:      'config/minion_pretrain_colab.yaml'
#   Mode B qlora finetune: 'config/minion_qlora_gpt2xl_colab.yaml'
#   Smoke test (< 1 min): 'config/smoke_test.yaml'

CONFIG_PATH = 'config/minion_pretrain_colab.yaml'

# Optional per-cell overrides (uncomment and edit as needed)
OVERRIDES = {
    'ckpt_dir':     DRIVE_CKPT_DIR,   # Always point to Drive
    # 'wandb_log':  True,             # Enable W&B logging
    # 'max_iters':  5000,             # Shorter run for testing
    # 'n_layer':    24,               # Larger model (A100 only)
    # 'n_embd':     1024,
    # 'compile':    False,            # Disable torch.compile for debugging
}

print(f'Config: {CONFIG_PATH}')
print(f'Overrides: {OVERRIDES}')

Config: config/minion_pretrain_colab.yaml
Overrides: {'ckpt_dir': '/content/drive/MyDrive/minion_ckpts'}


## Cell 7 — Start training

This cell runs the training loop.  It will:
- Auto-resume from the last checkpoint in Drive (if one exists)
- Print loss, VRAM usage, and MFU every `log_interval` steps
- Save a checkpoint every `eval_interval` steps
- Warn you ~15% before the session limit is reached

If the session disconnects, just re-run all cells — training will resume automatically.

In [ ]:
import os, sys

REPO_DIR = '/content/Minion'
if not os.path.exists(os.path.join(REPO_DIR, 'train.py')):
    os.system('git clone https://github.com/mlwithharsh/Minion.git ' + REPO_DIR)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

DRIVE_CKPT_DIR = '/content/drive/MyDrive/minion_ckpts'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

from train import train
train('config/minion_pretrain_colab.yaml', ckpt_dir=DRIVE_CKPT_DIR)


GPU: Tesla T4  |  VRAM: 14.6 GB  |  SM count: 40  |  Compute: 7.5
bf16 supported: True
Precision: bfloat16  |  AMP: True

Mode: pretrain
Loaded custom tokenizer: tokenizer/tokenizer.json  (vocab=32768)
Minion: 99.50M parameters
Gradient checkpointing enabled (12 blocks)
  decayed params:   99,483,648  (85 tensors)
  no-decay params:  19,200  (25 tensors)
bitsandbytes not installed, falling back to torch AdamW
Optimizer: torch AdamW (fused=True)
Loading streaming dataset 'HuggingFaceFW/fineweb-edu' (split=train)


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Loading streaming dataset 'HuggingFaceFW/fineweb-edu' (split=train)


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Compiling model with torch.compile() ... (~1 min, cache is lost on Colab restart)

Tokens per iteration: 65,536
Effective batch size: 64
Max iters: 50,000
Checkpoint dir: /content/drive/MyDrive/minion_ckpts



W0713 08:58:46.574000 4179 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2941: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(


step      0: train 10.5602  val 10.5609  lr 5.99e-07  vram 2.95GB  elapsed 0.03h
iter     0: loss 10.5641  128004.2ms/iter  mfu -100.0%  vram 4.12GB
iter    10: loss 9.8748  26840.5ms/iter  mfu 0.0%  vram 4.87GB
iter    20: loss 9.3959  27139.3ms/iter  mfu 0.0%  vram 4.87GB
iter    30: loss 9.1058  26783.7ms/iter  mfu 0.0%  vram 4.87GB
iter    40: loss 8.8908  26798.7ms/iter  mfu 0.0%  vram 4.87GB
iter    50: loss 8.5017  26839.4ms/iter  mfu 0.0%  vram 4.87GB
iter    60: loss 8.2616  26772.3ms/iter  mfu 0.0%  vram 4.87GB
iter    70: loss 8.0116  26779.1ms/iter  mfu 0.0%  vram 4.87GB
iter    80: loss 7.7567  26777.5ms/iter  mfu 0.0%  vram 4.87GB
iter    90: loss 7.6886  26786.7ms/iter  mfu 0.0%  vram 4.87GB
iter   100: loss 7.3439  26749.3ms/iter  mfu 0.0%  vram 4.87GB
iter   110: loss 7.1899  26769.0ms/iter  mfu 0.0%  vram 4.87GB
iter   120: loss 7.0571  26783.2ms/iter  mfu 0.0%  vram 4.87GB
iter   130: loss 7.1113  26718.9ms/iter  mfu 0.0%  vram 4.87GB
iter   140: loss 6.9434  26851.2

## Cell 8 — Quick generation test (after training)
Test the model directly in Colab before downloading to local machine.

In [ ]:
import os, sys, torch
sys.path.insert(0, '/content/minion')

PROMPT = 'The future of artificial intelligence is'  # ← edit this
CKPT   = os.path.join(DRIVE_CKPT_DIR, 'ckpt.pt')

if not os.path.exists(CKPT):
    print(f'No checkpoint found at {CKPT} — train first (Cell 7)')
else:
    from inference.quantize import load_quantized_model
    from inference.generate import generate_text

    model, tokenizer, cfg = load_quantized_model(
        ckpt_path = CKPT,
        quantize  = True,
        device    = 'cuda',
    )

    text = generate_text(
        model          = model,
        tokenizer      = tokenizer,
        cfg            = cfg,
        prompt         = PROMPT,
        max_new_tokens = 200,
        temperature    = 0.8,
        top_k          = 50,
        device         = 'cuda',
    )

    print('=== Generated text ===')
    print(PROMPT + text)

## Cell 9 — Download checkpoint to local machine

**Option A** (recommended): Use `rclone` on your local machine:
```bash
rclone sync "gdrive:minion_ckpts" ./checkpoints
```

**Option B**: Download directly from Colab (works for small checkpoints):

In [ ]:
# Download ckpt.pt directly from Colab to your browser
from google.colab import files
import os

CKPT = os.path.join(DRIVE_CKPT_DIR, 'ckpt.pt')
if os.path.exists(CKPT):
    size_mb = os.path.getsize(CKPT) / 1024**2
    print(f'Downloading ckpt.pt ({size_mb:.0f} MB) ...')
    files.download(CKPT)
else:
    print(f'No checkpoint at {CKPT}')